# L02 · One Lookup Structure for "If I Am Here and Do This"

*Build-Up Series · Lab 2 of 15 · standard library only · saves `rule_tables.py` and `floor_tables.py` · the Combine section at the end needs `grid_pieces.py` from L01 in the same folder*

**How to work through this notebook.** Type each code cell yourself, run it, and compare your output with the output printed here. Each code cell is small, and each is followed by an explanation. **Try it** boxes are guided experiments: copy your file, make the named change, look at the result, then go back to the original.

In L01 you learned that nested data is built from three layers: dicts, lists and tuples. This lab uses that to solve a very common problem: **storing rules**. By the end, a whole "world" of rules (what can happen, how likely, at what cost) lives in one value that you can check, query and edit without touching any code.

## 1 · The problem

A small company ships parcels. A parcel moves through states: it is **received**, then **packed**, then **shipped**, and finally **delivered**. But the world is messy:

* packing sometimes fails and has to be retried,
* you can choose how to ship (cheap ground or pricey express), and each has a different chance of the parcel getting lost,
* a delivery can bounce, sending the parcel back to the warehouse.

Each step has a fee. The natural first attempt is a ladder of `if` statements. Here it is; read it and notice how it feels.

In [1]:
def messy_outcomes(state, action):
    """Return a list of (chance, destination_state, fee) for doing `action` in `state`."""
    if state == "received":
        if action == "pack":
            return [(0.95, "packed", 2.0), (0.05, "received", 2.0)]
        elif action == "refuse":
            return [(1.0, "returned", 1.0)]
    elif state == "packed":
        if action == "ship_ground":
            return [(0.97, "shipped", 4.0), (0.03, "lost", 4.0)]
        elif action == "ship_express":
            return [(0.99, "shipped", 9.0), (0.01, "lost", 9.0)]
    elif state == "shipped":
        if action == "deliver":
            return [(0.90, "delivered", 1.0), (0.10, "returned", 1.0)]
    elif state == "returned":
        if action == "restock":
            return [(1.0, "received", 0.5)]
    return []   # "delivered", "lost", or an action that does not exist here


print(messy_outcomes("packed", "ship_ground"))   # a real action
print(messy_outcomes("packed", "ship_grond"))    # a typo
print(messy_outcomes("lost", "anything"))        # a dead-end state

[(0.97, 'shipped', 4.0), (0.03, 'lost', 4.0)]
[]
[]


**What happened.** The first call returns the two possible outcomes of shipping by ground: with chance 0.97 the parcel becomes `shipped`, with chance 0.03 it becomes `lost`, and either way the fee is 4.0. It works. The trouble is what it cannot do:

1. **The typo and the dead end look identical.** Both return `[]`. A misspelt action silently behaves like "nothing can happen", which is how bugs hide.
2. **You cannot ask the ladder a question.** "Which actions are allowed in `packed`?" has no answer unless you call the function with every action name you can think of.
3. **You cannot check it.** Do the chances in each outcome list add up to 1? You would have to read every line by eye.

**Analogy.** The ladder is like instructions that exist only as a spoken conversation with a clerk: you can ask one specific question at a time, but you can never see the whole rulebook. We want the rulebook itself, laid out so we can read it, check it and ask it questions.

## 2 · Tiny version (check it with a pencil)

Start with two states and one action, small enough to verify on paper. A switch is a bit flaky: flipping it from `off` works 90% of the time.

**Step 1: one outcome is a tuple with three parts.**

In [2]:
one_outcome = (0.9, "on", 1.0)           # (chance, destination state, fee)

chance, destination, fee = one_outcome   # unpacking: three names, three parts
print(chance)
print(destination)
print(fee)

0.9
on
1.0


**What happened.** An outcome answers three questions at once: *how likely*, *where do I end up*, *what does it cost*. A **tuple** is the right container because it is a small, fixed-size record of related facts (L01 introduced tuples as street addresses; this is the same idea with three parts instead of two). **Unpacking** (`chance, destination, fee = one_outcome`) gives each part its own descriptive name in one line.

**Step 2: one action has a list of possible outcomes.**

In [3]:
flip_outcomes_from_off = [(0.9, "on", 1.0), (0.1, "off", 1.0)]

total_chance = 0.0
for chance, destination, fee in flip_outcomes_from_off:   # unpack straight in the loop header
    total_chance += chance

print(total_chance)

1.0


**What happened.** An action that can go several ways is a **list** of outcomes. The chances must add up to 1, because *something* always happens: 0.9 + 0.1 = 1. Pencil check: ✓.

**Step 3: one state has several actions, found by name.**

In [4]:
actions_from_off = {"flip": [(0.9, "on", 1.0), (0.1, "off", 1.0)]}

print(list(actions_from_off))               # a dict listed as a list gives its keys
print(actions_from_off["flip"])

['flip']
[(0.9, 'on', 1.0), (0.1, 'off', 1.0)]


**What happened.** Actions are looked up by *name*, so they go in a **dict** from action name to outcome list. Listing a dict gives its keys, so `list(actions_from_off)` answers "which actions can I take here?", the question the ladder could not answer.

**Step 4: the whole world is a dict of states.**

In [5]:
tiny_table = {
    "off": {"flip": [(0.9, "on", 1.0), (0.1, "off", 1.0)]},
    "on":  {"flip": [(1.0, "off", 1.0)]},
}

# Read it with a path, exactly like L01: state, then action, then outcome number, then part.
print(tiny_table["off"]["flip"][0][1])   # destination of the first outcome

on


**What happened.** Three layers of the nested-data vocabulary from L01, stacked: a **dict** of states, each holding a **dict** of actions, each holding a **list** of outcome **tuples**. The path `tiny_table["off"]["flip"][0][1]` reads: state `off`, action `flip`, first outcome, second part (the destination): `on`.

Pencil check: from `off`, flipping works with chance 0.9 and leads to `on`. ✓

## 3 · Grow it: the real parcel rules

Now the same shape, but with the real rules. Type this table in; it is long, but every line is one rule.

In [6]:
parcel_table = {
    "received": {
        "pack":   [(0.95, "packed", 2.0), (0.05, "received", 2.0)],
        "refuse": [(1.0, "returned", 1.0)],
    },
    "packed": {
        "ship_ground":  [(0.97, "shipped", 4.0), (0.03, "lost", 4.0)],
        "ship_express": [(0.99, "shipped", 9.0), (0.01, "lost", 9.0)],
    },
    "shipped": {
        "deliver": [(0.90, "delivered", 1.0), (0.10, "returned", 1.0)],
    },
    "returned": {
        "restock": [(1.0, "received", 0.5)],
    },
    "delivered": {},   # dead end: nothing more can happen
    "lost": {},        # dead end
}

# Does the table say exactly what the ladder said? Compare every (state, action) pair.
same_as_ladder = all(
    messy_outcomes(state, action) == outcomes
    for state, actions in parcel_table.items()
    for action, outcomes in actions.items()
)
print(same_as_ladder)

True


**What happened.** The whole rulebook is one value. The last lines check that it agrees with the ladder. Read that generator expression slowly, because it is a pattern you will meet constantly: `for state, actions in ...items()` loops over the states; the second `for action, outcomes in actions.items()` loops over each state's actions; `all(...)` is `True` only if every comparison is `True`. Two `for` clauses in one comprehension read left to right just like two nested loops would.

**Step 5: lookups that explain themselves.**

In [7]:
def outcomes_for(table, state, action):
    """List of (chance, destination, fee) for doing `action` in `state`."""
    if state not in table:
        raise ValueError(f"unknown state {state!r}")
    if action not in table[state]:
        raise ValueError(
            f"state {state!r} has no action {action!r}; legal actions: {list(table[state])}")
    return table[state][action]


def legal_actions(table, state):
    """Names of the actions allowed in `state` (empty for a dead end)."""
    return list(table[state])


print(outcomes_for(parcel_table, "packed", "ship_ground"))
print(legal_actions(parcel_table, "received"))

try:
    outcomes_for(parcel_table, "packed", "ship_grond")   # the same typo as before
except ValueError as problem:
    print("ValueError:", problem)

[(0.97, 'shipped', 4.0), (0.03, 'lost', 4.0)]
['pack', 'refuse']
ValueError: state 'packed' has no action 'ship_grond'; legal actions: ['ship_ground', 'ship_express']


**What happened.** Two functions, one line of real work each. The important change is in the error: the typo that silently returned `[]` in the ladder now stops with a message that names the bad action *and lists the legal ones*. A dead-end state (`legal_actions(parcel_table, "lost")` is `[]`) is now clearly different from a mistake. Good structure lets you separate "nothing can happen" from "you asked wrongly".

**Step 6: checking a table before trusting it.**

First, a floating-point surprise that decides how we compare chances:

In [8]:
import math

print(0.1 + 0.2 == 0.3)                                # surprise!
print(0.1 + 0.2)                                       # the leftover sliver
print(math.isclose(0.1 + 0.2, 0.3, abs_tol=1e-9))      # close enough? yes

False
0.30000000000000004
True


**What happened.** Computers store most decimals like 0.1 as the nearest number they can represent in binary, which is very slightly off, so `0.1 + 0.2` is `0.30000000000000004` and not equal to `0.3`. The rule: **never test decimal numbers with `==`; test whether they are close**. `math.isclose(a, b, abs_tol=1e-9)` asks "are these within one billionth of each other?". We will use it to check that chances add up to 1.

**Analogy.** Measuring a table with a ruler: you never expect exactly 120 cm to infinite precision; you ask whether it is close enough for the job.

Next, a gotcha about copying nested data, which we need before we can safely break a table on purpose:

In [9]:
import copy

original_table = {"a": {"go": [(1.0, "a", 0.0)]}}

shallow_copy = dict(original_table)         # new OUTER dict, but the SAME inner dicts inside it
shallow_copy["a"]["wait"] = [(1.0, "a", 0.0)]
print("original changed too:", "wait" in original_table["a"])

original_table = {"a": {"go": [(1.0, "a", 0.0)]}}
deep_copy = copy.deepcopy(original_table)   # copies every layer all the way down
deep_copy["a"]["wait"] = [(1.0, "a", 0.0)]
print("deep copy left the original alone:", "wait" not in original_table["a"])

original changed too: True
deep copy left the original alone: True


**What happened.** A *shallow* copy builds a new outer container but reuses the very same inner containers, so editing an inner dict through the copy also edits the original. `copy.deepcopy` copies every layer. **Rule of thumb: when a structure is nested, copy it with `deepcopy` if you plan to edit the copy.**

**Analogy.** A shallow copy is a new filing cabinet whose folders are the same physical folders as the old cabinet's. Changing a paper in one cabinet changes it in the other.

Now the validator. It returns a *list of problems* instead of stopping at the first one, so you can fix everything in one pass.

In [10]:
def validate_table(table):
    """Return a list of human-readable problems; an empty list means the table is sound."""
    problems = []
    for state, actions in table.items():
        for action, outcomes in actions.items():
            where = f"{state!r}/{action!r}"
            if not outcomes:
                problems.append(f"{where}: no outcomes listed")
                continue
            total_chance = 0.0
            for chance, destination, fee in outcomes:
                if not 0.0 <= chance <= 1.0:
                    problems.append(f"{where}: chance {chance} is outside 0..1")
                if destination not in table:
                    problems.append(f"{where}: destination {destination!r} is not a state")
                if fee < 0:
                    problems.append(f"{where}: negative fee {fee}")
                total_chance += chance
            if not math.isclose(total_chance, 1.0, abs_tol=1e-9):
                problems.append(f"{where}: chances add up to {total_chance}, not 1")
    return problems


print(validate_table(parcel_table))

[]


In [11]:
# Break a DEEP copy three different ways, then see whether the validator notices each.
broken_table = copy.deepcopy(parcel_table)
broken_table["shipped"]["deliver"][0] = (0.80, "delivered", 1.0)     # chances no longer add to 1
broken_table["packed"]["ship_ground"][1] = (0.03, "lots", 4.0)       # a misspelt destination
broken_table["received"]["refuse"] = [(1.0, "returned", -1.0)]       # a negative fee

for problem in validate_table(broken_table):
    print(problem)

'received'/'refuse': negative fee -1.0
'packed'/'ship_ground': destination 'lots' is not a state
'shipped'/'deliver': chances add up to 0.9, not 1


**What happened.** The real table passes (an empty problem list). The deliberately broken copy gets three separate complaints, each naming the exact state and action. Something the ladder could never offer: **rules you can audit automatically.**

**Try it (guided experiment).** Copy your file. In the copy, change every outcome from a tuple `(chance, destination, fee)` to a small dict `{"chance": 0.95, "to": "packed", "fee": 2.0}`, and change `validate_table` to read `outcome["chance"]` and so on. Run it on both tables and compare the readability of the two versions. Write down one thing that got clearer (hint: the names inside the data) and one thing that got more verbose (hint: look at the table literal). Then return to the original file. Do this once you can explain why the validator needs to unpack three parts from each outcome.

**Step 7: walking the table.**

In [12]:
def reachable_states(table, start):
    """Every state you could possibly end up in, starting from `start`."""
    seen = {start}        # a SET: holds each item at most once, and answers "is it in there?" fast
    to_visit = [start]    # states whose actions we have not looked at yet
    while to_visit:       # keep going until the to-do list is empty
        state = to_visit.pop()                        # take one state off the to-do list
        for outcomes in table[state].values():
            for _chance, destination, _fee in outcomes:   # _name means "I will not use this"
                if destination not in seen:
                    seen.add(destination)
                    to_visit.append(destination)
    return seen


print(sorted(reachable_states(parcel_table, "received")))
print(sorted(reachable_states(parcel_table, "delivered")))

['delivered', 'lost', 'packed', 'received', 'returned', 'shipped']
['delivered']


**What happened.** From `received` every state is reachable, which matches a walk by hand: `received` → `packed` or `returned`; `packed` → `shipped` or `lost`; `shipped` → `delivered` or `returned`; and `returned` leads back to `received`. From the dead end `delivered` you can reach only `delivered` itself.

The function keeps a **set** of states already found (so nothing is visited twice, which also stops it looping forever around the `received`–`returned` circle) and a **to-do list** of states still to look inside. `while to_visit:` repeats until the to-do list is empty.

**Analogy.** Exploring a building with a pile of unopened doors: take a door from the pile, open it, write any new rooms on the pile, and never revisit a room you have already marked on your map.

**Step 8: building the table from a text file.**

Real rules usually arrive as text written by a person, not as a Python literal. Reuse the skill from L01: text in, nested data out.

In [13]:
rules_text = """
received --pack--> 0.95 packed $2.0 ; 0.05 received $2.0
received --refuse--> 1.0 returned $1.0
packed --ship_ground--> 0.97 shipped $4.0 ; 0.03 lost $4.0
packed --ship_express--> 0.99 shipped $9.0 ; 0.01 lost $9.0
shipped --deliver--> 0.9 delivered $1.0 ; 0.1 returned $1.0
returned --restock--> 1.0 received $0.5
delivered
lost
"""


def parse_rules(rules_text):
    """Rule lines like 'state --action--> chance destination $fee ; ...'  ->  nested table."""
    table = {}
    for rule_line in rules_text.strip().splitlines():
        rule_line = rule_line.strip()
        if not rule_line:
            continue
        if "-->" not in rule_line:
            table.setdefault(rule_line, {})          # a bare state name: a dead end
            continue
        left_text, _, outcomes_text = rule_line.partition("-->")
        state, _, action = left_text.strip().partition("--")
        outcomes = []
        for outcome_text in outcomes_text.split(";"):
            chance_text, destination, fee_text = outcome_text.split()
            outcomes.append((float(chance_text), destination, float(fee_text.lstrip("$"))))
        table.setdefault(state.strip(), {})[action.strip()] = outcomes
    return table


print(parse_rules(rules_text) == parcel_table)

True


**What happened.** Each rule line is cut at `-->`: the left part holds the state and action (cut again at `--`), and the right part holds one or more outcomes separated by `;`. `str.split()` with no argument cuts on any run of spaces, so each outcome becomes exactly three words, which unpack into names. `float("0.95")` turns text into a number, and `lstrip("$")` removes the leading dollar sign first. The final check shows that the table built from text is *identical* to the one we typed by hand.

## 4 · Aha: one structure answers every question, and rules are data

Look at what we can now ask of one value, with no new machinery:

In [14]:
# 1. Which actions can I take in "packed"?
print(list(parcel_table["packed"]))

# 2. Which states are dead ends?
print([state for state, actions in parcel_table.items() if not actions])

# 3. What is the worst fee each "packed" action could cost me?
for action, outcomes in parcel_table["packed"].items():
    print(action, max(fee for _chance, _destination, fee in outcomes))

# 4. Which (state, action) pairs could end in "lost"?
print([(state, action)
       for state, actions in parcel_table.items()
       for action, outcomes in actions.items()
       if any(destination == "lost" for _chance, destination, _fee in outcomes)])

['ship_ground', 'ship_express']
['delivered', 'lost']
ship_ground 4.0
ship_express 9.0
[('packed', 'ship_ground'), ('packed', 'ship_express')]


**Four different questions, four one-line answers, one structure.** Now the second half of the aha: **changing a rule means editing data, not code.** Make express shipping cost 12.0 and ask question 3 again, with no function changed:

In [15]:
pricier_table = copy.deepcopy(parcel_table)    # a deep copy: we will edit inner layers
pricier_table["packed"]["ship_express"] = [(0.99, "shipped", 12.0), (0.01, "lost", 12.0)]

for action, outcomes in pricier_table["packed"].items():
    print(action, max(fee for _chance, _destination, fee in outcomes))

print(validate_table(pricier_table))   # still a sound table

ship_ground 4.0
ship_express 12.0
[]


**The aha.** The whole world fits in one lookup structure, and the code that asks questions of it never changes when the rules do. That is why so many programs are built this way: the rules live in a table that people (or other programs) can read, check and edit, while a small stable set of functions does the asking.

**Where else this shows up.** The same shape, "(where I am, what I do) leads to a list of (chance, new place, cost)", is how you would describe an order workflow like this one, the moves of a text-adventure game (room, command to room), the phases of a traffic-light controller, and the allowed steps of a network protocol. Later labs will ask things of tables like this one: what is the cheapest way through, what will happen on average, which action should I pick.

## 5 · Your turn

The same small checker as in L01, reused here so you can see its result immediately.

In [16]:
def check(label, function, cases):
    """cases is a list of (arguments_tuple, expected_result) pairs."""
    for arguments, expected in cases:
        try:
            got = function(*arguments)
        except Exception as problem:
            print(f"{label}: not yet (raised {type(problem).__name__}: {problem})")
            return
        if got is None:
            print(f"{label}: not yet")
            return
        if got != expected:
            print(f"{label}: FAIL  arguments={arguments!r}  expected={expected!r}  got={got!r}")
            return
    print(f"{label}: PASS")

**Challenge 1: the chance of ending somewhere.** Write `chance_of_ending_in(table, state, action, target)`: the total chance that doing `action` in `state` ends in the state `target`. By hand: `ship_ground` from `packed` ends in `lost` with chance 0.03, in `shipped` with 0.97, and in `delivered` with 0.

*Hint:* a `sum` over the outcomes whose destination equals `target`. A `sum` over nothing is 0.

**Challenge 2: add an action safely.** Write `add_action(table, state, action, outcomes)` that returns a **new** table with the new action added, leaving the original untouched. It must raise `ValueError` if the state is unknown, if the action already exists, or if the result fails `validate_table`.

*Hint:* `copy.deepcopy` first (L02 Step 6), add to the copy, run the validator, and raise with the problems joined into one message.

**Challenge 3: who can reach the goal?** Write `states_that_can_reach(table, target)` that returns a **sorted list** of every state from which `target` can be reached, including `target` itself. By hand for `"delivered"`: `shipped` (deliver), `packed` (ships), `received` (packs) and `returned` (restocks back to `received`) can; `lost` cannot.

*Hint:* start with `{target}`, then repeatedly add any state that has an outcome landing in the set, until a full pass adds nothing new. This is `reachable_states` run backwards.

Replace each `pass` with your answer.

In [17]:
def chance_of_ending_in(table, state, action, target):
    pass   # replace this line with your answer


def add_action(table, state, action, outcomes):
    pass   # replace this line with your answer


def states_that_can_reach(table, target):
    pass   # replace this line with your answer


def add_action_probe(table):
    """Exercises add_action: result, original untouched, and a bad table rejected."""
    new_outcomes = [(0.8, "shipped", 6.0), (0.2, "lost", 6.0)]
    new_table = add_action(table, "packed", "ship_drone", new_outcomes)
    if new_table is None:
        return None
    try:
        add_action(table, "packed", "ship_bad", [(0.5, "shipped", 1.0)])   # chances add to 0.5
        verdict = "accepted"
    except ValueError:
        verdict = "rejected"
    return (new_table["packed"]["ship_drone"], "ship_drone" in table["packed"], verdict)


def run_all_checks():
    check("chance_of_ending_in", chance_of_ending_in, [
        ((parcel_table, "packed", "ship_ground", "lost"), 0.03),
        ((parcel_table, "packed", "ship_ground", "shipped"), 0.97),
        ((parcel_table, "packed", "ship_ground", "delivered"), 0.0)])
    check("add_action", add_action_probe, [
        ((parcel_table,), ([(0.8, "shipped", 6.0), (0.2, "lost", 6.0)], False, "rejected"))])
    check("states_that_can_reach", states_that_can_reach, [
        ((parcel_table, "delivered"), ["delivered", "packed", "received", "returned", "shipped"])])


run_all_checks()

chance_of_ending_in: not yet
add_action: not yet
states_that_can_reach: not yet


### Solutions (try first)

Each replaces the stub of the same name. Run `run_all_checks()` again after you define them.

In [18]:
def chance_of_ending_in(table, state, action, target):
    outcomes = outcomes_for(table, state, action)
    return sum(chance for chance, destination, _fee in outcomes if destination == target)


def add_action(table, state, action, outcomes):
    if state not in table:
        raise ValueError(f"unknown state {state!r}")
    if action in table[state]:
        raise ValueError(f"state {state!r} already has action {action!r}")
    new_table = copy.deepcopy(table)            # never edit the caller's table
    new_table[state][action] = list(outcomes)
    problems = validate_table(new_table)
    if problems:
        raise ValueError("; ".join(problems))
    return new_table


def states_that_can_reach(table, target):
    can_reach = {target}
    something_changed = True
    while something_changed:                    # repeat full passes until a pass adds nothing
        something_changed = False
        for state, actions in table.items():
            if state in can_reach:
                continue
            for outcomes in actions.values():
                if any(destination in can_reach for _chance, destination, _fee in outcomes):
                    can_reach.add(state)
                    something_changed = True
                    break                       # one landing in the set is enough for this state
    return sorted(can_reach)


run_all_checks()

chance_of_ending_in: PASS
add_action: PASS
states_that_can_reach: PASS


**Why Challenge 3 matters later.** You just solved "which states can reach the goal?" by working *backwards* from the goal, repeating until nothing changes. Remember the feeling of that backward sweep: L05 uses the same idea to fill in a whole table of answers.

## 6 · Piece card

| Field | Entry |
|---|---|
| **Name** | `rule_tables` (`outcomes_for`, `legal_actions`, `validate_table`, `reachable_states`, `parse_rules`, `chance_of_ending_in`, `add_action`, `states_that_can_reach`) |
| **Input** | a **table**: dict of state to dict of action to list of `(chance, destination, fee)` tuples; or rule text for `parse_rules` |
| **Output** | lookups (outcome lists, action names), a list of problems (empty means sound), sets and sorted lists of states |
| **Assumes** | states are hashable (strings, tuples); every destination is itself a key of the table; the chances of each action add to 1 within 1e-9; fees are not negative; a dead end has an empty dict of actions |
| **Three versions in this lab** | nested dict (L02 shape), flat dict keyed by `(state, action)`, a class wrapping the nested dict |
| **Swap rule** | any version may replace another if `outcomes_for` returns the same list for every `(state, action)` pair |
| **Used later by** | L03, L04, L05, L07, L08, A1, A2 |

Save it as a module file, then compare the module with the notebook, exactly as in L01.

In [19]:
module_source = r'''"""rule_tables.py  (L02 piece card)

Name:        rule_tables  (outcomes_for, legal_actions, validate_table, reachable_states,
             parse_rules, chance_of_ending_in, add_action, states_that_can_reach)
Input:       table = dict of state -> dict of action -> list of (chance, destination, fee)
Output:      lookups, a list of problems (empty = sound), sets / sorted lists of states
Assumes:     states hashable; every destination is a key; chances per action sum to 1 (1e-9);
             fees >= 0; dead end = empty dict of actions
Versions:    nested dict / flat (state, action) dict / class wrapper
Swap rule:   same outcomes_for result for every (state, action)
Used later:  L03, L04, L05, L07, L08, A1, A2
"""
import copy
import math


def outcomes_for(table, state, action):
    if state not in table:
        raise ValueError(f"unknown state {state!r}")
    if action not in table[state]:
        raise ValueError(
            f"state {state!r} has no action {action!r}; legal actions: {list(table[state])}")
    return table[state][action]


def legal_actions(table, state):
    return list(table[state])


def validate_table(table):
    problems = []
    for state, actions in table.items():
        for action, outcomes in actions.items():
            where = f"{state!r}/{action!r}"
            if not outcomes:
                problems.append(f"{where}: no outcomes listed")
                continue
            total_chance = 0.0
            for chance, destination, fee in outcomes:
                if not 0.0 <= chance <= 1.0:
                    problems.append(f"{where}: chance {chance} is outside 0..1")
                if destination not in table:
                    problems.append(f"{where}: destination {destination!r} is not a state")
                if fee < 0:
                    problems.append(f"{where}: negative fee {fee}")
                total_chance += chance
            if not math.isclose(total_chance, 1.0, abs_tol=1e-9):
                problems.append(f"{where}: chances add up to {total_chance}, not 1")
    return problems


def reachable_states(table, start):
    seen = {start}
    to_visit = [start]
    while to_visit:
        state = to_visit.pop()
        for outcomes in table[state].values():
            for _chance, destination, _fee in outcomes:
                if destination not in seen:
                    seen.add(destination)
                    to_visit.append(destination)
    return seen


def parse_rules(rules_text):
    table = {}
    for rule_line in rules_text.strip().splitlines():
        rule_line = rule_line.strip()
        if not rule_line:
            continue
        if "-->" not in rule_line:
            table.setdefault(rule_line, {})
            continue
        left_text, _, outcomes_text = rule_line.partition("-->")
        state, _, action = left_text.strip().partition("--")
        outcomes = []
        for outcome_text in outcomes_text.split(";"):
            chance_text, destination, fee_text = outcome_text.split()
            outcomes.append((float(chance_text), destination, float(fee_text.lstrip("$"))))
        table.setdefault(state.strip(), {})[action.strip()] = outcomes
    return table


def chance_of_ending_in(table, state, action, target):
    outcomes = outcomes_for(table, state, action)
    return sum(chance for chance, destination, _fee in outcomes if destination == target)


def add_action(table, state, action, outcomes):
    if state not in table:
        raise ValueError(f"unknown state {state!r}")
    if action in table[state]:
        raise ValueError(f"state {state!r} already has action {action!r}")
    new_table = copy.deepcopy(table)
    new_table[state][action] = list(outcomes)
    problems = validate_table(new_table)
    if problems:
        raise ValueError("; ".join(problems))
    return new_table


def states_that_can_reach(table, target):
    can_reach = {target}
    something_changed = True
    while something_changed:
        something_changed = False
        for state, actions in table.items():
            if state in can_reach:
                continue
            for outcomes in actions.values():
                if any(destination in can_reach for _chance, destination, _fee in outcomes):
                    can_reach.add(state)
                    something_changed = True
                    break
    return sorted(can_reach)
'''

with open("rule_tables.py", "w", encoding="utf-8") as module_file:
    module_file.write(module_source)

import importlib
import rule_tables
importlib.reload(rule_tables)

print("module matches notebook:",
      rule_tables.parse_rules(rules_text) == parcel_table
      and rule_tables.validate_table(broken_table) == validate_table(broken_table)
      and rule_tables.reachable_states(parcel_table, "received") == reachable_states(parcel_table, "received")
      and rule_tables.states_that_can_reach(parcel_table, "delivered") == states_that_can_reach(parcel_table, "delivered"))

module matches notebook: True


## 7 · Swap test, then Combine

**The swap.** The piece "a table of rules" can be stored three genuinely different ways:

* **A: nested dict** (what we built): `table[state][action]`.
* **B: flat dict keyed by `(state, action)` tuples**: `table[(state, action)]`.
* **C: a class** that wraps the nested dict and offers named methods.

Form B depends on a fact from L01: a tuple can be a dict key, and a list cannot. First, to see why:

In [20]:
print({(3, 5): "ok"}[(3, 5)])              # a tuple key works

try:
    print({[3, 5]: "no"})                  # a list key does not
except TypeError as problem:
    print("TypeError:", problem)

ok
TypeError: unhashable type: 'list'


**What happened.** A dict key must be **hashable**: Python must be able to turn it into a fixed fingerprint so it can find the entry again quickly. Anything that can change after it is stored (a list) could change its fingerprint, so lists are refused. Tuples cannot change, so they are allowed. This is why an `(row, column)` address makes a fine key.

Now the rival versions, answering the same questions as version A:

In [21]:
# Version B: one flat dict. The key is the pair (state, action).
flat_table = {
    (state, action): outcomes
    for state, actions in parcel_table.items()
    for action, outcomes in actions.items()
}


def outcomes_for_flat(flat, state, action):
    return flat[(state, action)]


def legal_actions_flat(flat, state):
    # No direct way to ask "all actions of this state": scan EVERY key and keep the matching ones.
    return [action for (state_name, action) in flat if state_name == state]


# Version C: a class that owns the nested table and exposes named methods.
class RuleTable:
    def __init__(self, nested_table):
        self._nested_table = nested_table          # the leading underscore says "internal"

    def outcomes(self, state, action):
        return self._nested_table[state][action]

    def legal_actions(self, state):
        return list(self._nested_table[state])


class_table = RuleTable(parcel_table)

pairs = [(state, action) for state, actions in parcel_table.items() for action in actions]
print("same outcomes for every pair:",
      all(outcomes_for(parcel_table, s, a) == outcomes_for_flat(flat_table, s, a) == class_table.outcomes(s, a)
          for s, a in pairs))
print("same legal actions for every state:",
      all(legal_actions(parcel_table, s) == legal_actions_flat(flat_table, s) == class_table.legal_actions(s)
          for s in parcel_table))

same outcomes for every pair: True
same legal actions for every state: True


**What happened.** All three versions give identical answers on every question we can ask of the parcel table, so the swap rule from the piece card holds. Where they differ is in *cost* and *feel*:

* In B, `legal_actions_flat` has to scan every key in the table, because the flat shape has no per-state grouping.
* In C, the data is the same as A, but callers say `table.outcomes(...)` and never see the dicts inside.

Let us measure the cost of B's scan on a big generated table: 2,000 states with 4 actions each.

In [22]:
import time


def make_big_table(number_of_states, actions_per_state):
    """A synthetic table: every action has two equally likely outcomes."""
    big_table = {}
    for state_number in range(number_of_states):
        actions = {}
        for action_number in range(actions_per_state):
            first_destination = f"s{(state_number + action_number + 1) % number_of_states}"
            second_destination = f"s{(state_number * 2 + action_number) % number_of_states}"
            actions[f"a{action_number}"] = [(0.5, first_destination, 1.0), (0.5, second_destination, 1.0)]
        big_table[f"s{state_number}"] = actions
    return big_table


big_nested = make_big_table(2000, 4)
big_flat = {(state, action): outcomes
            for state, actions in big_nested.items() for action, outcomes in actions.items()}
queried_states = [f"s{number}" for number in range(300)]   # 300 "which actions can I take?" questions


def seconds_to_answer(function, table):
    start = time.perf_counter()
    answers = [function(table, state) for state in queried_states]
    return time.perf_counter() - start, answers


nested_seconds, nested_answers = seconds_to_answer(legal_actions, big_nested)
flat_seconds, flat_answers = seconds_to_answer(legal_actions_flat, big_flat)

print(f"nested: {nested_seconds * 1000:.2f} ms for {len(queried_states)} questions")
print(f"flat:   {flat_seconds * 1000:.2f} ms for {len(queried_states)} questions")
print("same answers:", nested_answers == flat_answers)

nested: 0.21 ms for 300 questions
flat:   105.15 ms for 300 questions
same answers: True


**Reading the result honestly.** The two timing lines above were measured on the machine that ran this notebook and will differ on yours. What to take from them is the *shape* of the difference, not the exact numbers: nested lookups go straight to one state's entry, while the flat form looks at every key in the table for each question, so its cost depends on the size of the whole table. (That scaling follows from how the code works; this run measured only one table size, so treat it as an explanation, not a measured curve.) The flat form is the better fit when you mostly ask "what happens for exactly this pair?" (a direct lookup in both); the nested form is better when you also ask per-state questions. Choosing a shape is choosing which questions are cheap.

**Combine: a floor with a slippery surface.** Combine is the one place earlier labs may be imported. L01's `grid_pieces.py` gives us floor plans; this lab's tables can describe *moving across one*. Imagine the bakery floor has just been mopped. When you try to step in a direction, you move as intended with chance 0.8, but you slide sideways with chance 0.1 each way. If the square you would slide to is blocked, you stay where you are. Every attempt costs 1.0. Squares are the states (addresses as tuple keys) and directions are the actions.

This cell needs `grid_pieces.py`, written by L01, to be in the same folder.

In [23]:
from grid_pieces import parse_floor_plan, neighbors_of, kind_at, BLOCKED_KINDS

bakery_text = """
name: Corner Bakery
legend: # wall | . floor | D door | S start | E exit

##########
#S...#...#
#.##.#.#.#
#.#..D.#E#
#.#.####.#
#...D...#
#########
"""
combine_plan = parse_floor_plan(bakery_text)

# For each direction, the two sideways directions you might slide into.
SIDEWAYS = {"up": ("left", "right"), "down": ("left", "right"),
            "left": ("up", "down"), "right": ("up", "down")}


def build_floor_table(plan, slip_chance=0.1):
    """Table for moving over a slippery floor: states are (row, column), actions are directions."""
    exit_squares = set(plan["markers"].get("exit", []))
    table = {}
    for row in range(plan["height"]):
        for column in range(plan["width"]):
            if kind_at(plan, row, column) in BLOCKED_KINDS:
                continue                              # walls and empty space are not places you can be
            address = (row, column)
            actions = {}
            if address not in exit_squares:           # reaching the exit ends the story: a dead end
                open_neighbors = neighbors_of(plan, row, column)
                for direction, (first_side, second_side) in SIDEWAYS.items():
                    chance_by_destination = {}
                    for move_direction, chance in [(direction, 1.0 - 2 * slip_chance),
                                                   (first_side, slip_chance),
                                                   (second_side, slip_chance)]:
                        # A blocked move leaves you where you are.
                        destination = open_neighbors.get(move_direction, address)
                        # Two different slides can land on the same square: add their chances.
                        chance_by_destination[destination] = (
                            chance_by_destination.get(destination, 0.0) + chance)
                    actions[direction] = [(chance, destination, 1.0)
                                          for destination, chance in chance_by_destination.items()]
            table[address] = actions
    return table


floor_table = build_floor_table(combine_plan)
start_square = combine_plan["markers"]["start"][0]
exit_square = combine_plan["markers"]["exit"][0]

print("table sound:", rule_tables.validate_table(floor_table) == [])
print("states in table:", len(floor_table))
print("exit reachable from start:", exit_square in rule_tables.reachable_states(floor_table, start_square))
print("legal actions at the start:", rule_tables.legal_actions(floor_table, start_square))
print("outcomes of 'right' at the start:", rule_tables.outcomes_for(floor_table, start_square, "right"))
print("outcomes of 'up' at the start:", rule_tables.outcomes_for(floor_table, start_square, "up"))

table sound: True
states in table: 27
exit reachable from start: True
legal actions at the start: ['up', 'down', 'left', 'right']
outcomes of 'right' at the start: [(0.8, (1, 2), 1.0), (0.1, (1, 1), 1.0), (0.1, (2, 1), 1.0)]
outcomes of 'up' at the start: [(0.9, (1, 1), 1.0), (0.1, (1, 2), 1.0)]


**What happened.** Two earlier pieces fed a third. `parse_floor_plan` and `neighbors_of` (L01) turned text into the geometry; `build_floor_table` turned that geometry into the L02 table shape; and then the L02 functions validated and queried it with no changes. The validator confirms that all the chances add up, even where two slides landed on the same square and their chances had to be merged.

Look at the last two lines and check them against the map. The start square `(1, 1)` has a wall above it and to its left. Moving `right` reaches `(1, 2)` with chance 0.8; a slide up hits the wall so you stay put (0.1), and a slide down reaches `(2, 1)` (0.1). Moving `up` is into a wall, so the intended move leaves you in place (0.8), the slide left is also blocked and leaves you in place (0.1 more, merged into one entry of 0.9), and only the slide right gets you anywhere, to `(1, 2)` (0.1). Compare with the printed lists.

Save this as a module too, so later labs can build the same table in one line:

In [24]:
floor_module_source = r'''"""floor_tables.py  (L02 Combine)

Name:        build_floor_table(plan, slip_chance=0.1)
Input:       a plan from grid_pieces.parse_floor_plan
Output:      a rule table (L02 shape): states are (row, column), actions are directions
Assumes:     exit squares are dead ends; a blocked move leaves you in place; each attempt costs 1.0
Used later:  L05, L07, A1, A2
"""
from grid_pieces import neighbors_of, kind_at, BLOCKED_KINDS

SIDEWAYS = {"up": ("left", "right"), "down": ("left", "right"),
            "left": ("up", "down"), "right": ("up", "down")}


def build_floor_table(plan, slip_chance=0.1):
    exit_squares = set(plan["markers"].get("exit", []))
    table = {}
    for row in range(plan["height"]):
        for column in range(plan["width"]):
            if kind_at(plan, row, column) in BLOCKED_KINDS:
                continue
            address = (row, column)
            actions = {}
            if address not in exit_squares:
                open_neighbors = neighbors_of(plan, row, column)
                for direction, (first_side, second_side) in SIDEWAYS.items():
                    chance_by_destination = {}
                    for move_direction, chance in [(direction, 1.0 - 2 * slip_chance),
                                                   (first_side, slip_chance),
                                                   (second_side, slip_chance)]:
                        destination = open_neighbors.get(move_direction, address)
                        chance_by_destination[destination] = (
                            chance_by_destination.get(destination, 0.0) + chance)
                    actions[direction] = [(chance, destination, 1.0)
                                          for destination, chance in chance_by_destination.items()]
            table[address] = actions
    return table
'''

with open("floor_tables.py", "w", encoding="utf-8") as module_file:
    module_file.write(floor_module_source)

import floor_tables
importlib.reload(floor_tables)
print("floor module matches notebook:", floor_tables.build_floor_table(combine_plan) == floor_table)

floor module matches notebook: True


**Before you go on, check yourself.** Without scrolling up: (1) what are the three parts of one outcome, (2) why must a flat table's keys be tuples and not lists, and (3) why do we use `copy.deepcopy` before editing a nested table? Once those are solid, a good experiment beyond the lab is to add a new action to the parcel table with your own `add_action`, then ask `states_that_can_reach` whether it changed which states can still reach `delivered`.